# 03. 모델 활용 방안

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json, pickle
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.data import load_raw, clean
from src.features import to_long, build_features
from src.viz import setup_plot_style, DOW_KR

setup_plot_style()
pd.set_option("display.max_columns", 50)
DATA, OUT = ROOT / C.DATA_PATH, ROOT / C.OUTPUT_DIR

required = ["predictions_test.csv", "run_info.json", C.FINAL_MODEL_FILE]
missing = [f for f in required if not (OUT / f).exists()]
if missing:
    raise FileNotFoundError(f"run.py 실행 결과가 없습니다: {missing}")

preds = pd.read_csv(OUT / "predictions_test.csv", parse_dates=["ts", "date"])
info = json.loads((OUT / "run_info.json").read_text(encoding="utf-8"))
selected = info["selected_model"]
with open(OUT / C.FINAL_MODEL_FILE, "rb") as f:
    model = pickle.load(f)
feat = build_features(to_long(clean(load_raw(DATA))[0]))
print("선정 모델:", selected)

## 1. 테스트 구간 예측 (1주)

In [ ]:
week = pd.date_range(preds["date"].min(), periods=7, freq="D")
w = preds[preds["date"].isin(week)]
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(w["ts"], w["kw"], color="black", lw=1, label="실측")
ax.plot(w["ts"], w[selected], color="tab:blue", lw=1, label=f"예측 ({selected})")
ax.set_xticks(week)
ax.set_xticklabels([f"{d:%m-%d} ({DOW_KR[d.dayofweek + 1]})" for d in week])
ax.set_ylabel("15분 수요 (kW)")
ax.legend(fontsize=8, loc="upper right")
plt.show()

## 2. 피크 감소 방안

전기요금 측면에서도 피크 구간을 줄이는 것은 중요하다. 한 달 동안 측정되는 약 2,880개의 15분 구간 중 가장 높은 전력 사용량이 해당 월의 최대수요전력, 즉 피크치가 된다. 또한 요금 적용 전력은 당월 피크치와 직전 12개월 동안 여름철(7~9월) 및 겨울철(12~2월)에 기록된 피크치 중 더 큰 값으로 결정된다.

이처럼 일시적인 전력 수요 급증이 장기간의 요금 부담으로 이어질 수 있으므로, 전력 수요가 높은 시간대의 생산 공정을 낮은 시간대로 분산해 피크 구간을 완화하는 것이 전기요금 절감에 효과적인 방안이 될 수 있다.

### 실제 전력 예측 그래프

In [ ]:
available_dates = sorted(preds["date"].dropna().unique())
TEST_DATE = pd.Timestamp(available_dates[1])

if TEST_DATE not in set(preds["date"].dropna().unique()):
    available_dates_text = [d.strftime("%Y-%m-%d") for d in available_dates]
    raise ValueError(f"TEST_DATE가 테스트 구간에 없습니다: {TEST_DATE:%Y-%m-%d}\n사용 가능한 날짜: {available_dates_text}")

day = preds.loc[preds["date"].eq(TEST_DATE)].sort_values("ts").copy()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(day["ts"], day[selected], color="tab:blue", lw=1.5, label=f"예측 ({selected})")
ax.set_title(f"테스트 구간 하루 예측 전력량: {TEST_DATE:%Y-%m-%d} ({DOW_KR[TEST_DATE.dayofweek + 1]})")
ax.set_xlabel("시간")
ax.set_ylabel("15분 수요 (kW)")
ax.xaxis.set_major_locator(plt.matplotlib.dates.HourLocator(interval=1))
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%H:%M"))
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8, loc="upper right")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# 15분 단위 예측 수요를 에너지로 환산한 하루 합계(kWh)입니다.
print(f"예측 하루 전력량: {day[selected].sum() * 0.25:,.2f} kWh")

예측 그래프를 보면 아침 시간대에는 전력 수요가 낮게 나타나며, 일반적인 근무 시간대인 09:00~18:00에는 비교적 높은 수준을 유지한 뒤 점차 감소하는 경향을 보인다. 점심시간인 12:00~13:00에는 전력 수요가 일시적으로 크게 감소한다.

### 활용 방안 1. 주간 시간대 사용량 분산

일별 최고 전력 수요는 주로 주간에 나타난다. 또한 수요가 일정하지 않고 특정 시점에 급격히 상승하거나 하락하므로, 수요가 낮은 시간대로 일부 사용량을 분산하면 피크 전력 수요를 낮출 수 있을 것으로 예상된다.

#### 점심 시간을 제외한 주간 시간대 전력 예측 그래프

In [ ]:
# 근무 시간대(09:00~18:00)만 표시하고 점심시간(12:00~13:00)은 제외합니다.
hour = day["ts"].dt.hour
workday = day.loc[hour.between(9, 18) & ~hour.eq(12)].copy()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(workday["ts"], workday[selected], color="tab:blue", lw=1.5, label=f"예측 ({selected})")
ax.set_title(f"근무 시간대 예측 전력량: {TEST_DATE:%Y-%m-%d} ({DOW_KR[TEST_DATE.dayofweek + 1]})")
ax.set_xlabel("시간")
ax.set_ylabel("15분 수요 (kW)")
ax.set_xlim(TEST_DATE + pd.Timedelta(hours=9), TEST_DATE + pd.Timedelta(hours=18))
ax.xaxis.set_major_locator(plt.matplotlib.dates.HourLocator(interval=1))
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%H:%M"))
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8, loc="upper right")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

#### 생산 시점 조정 방안

09:00~18:00 구간의 예측 전력량을 기준으로, 상한선 위의 초과 전력량과 상한선 아래의 여유 전력량이 같아지는 지점을 찾으면 전력 수요를 균형 있게 나눌 수 있는 상한선을 설정할 수 있다. 또한 예측 곡선과 상한선이 만나는 시각을 함께 확인하면 기준보다 전력 수요가 높은 시간대와 낮은 시간대를 구분할 수 있다.

상한선을 찾기 위해서 목표 시간 대의 최솟값과 최댓값을 찾은 뒤 이 둘 사이에서 이진 탐색을 실시한다.

이 정보를 활용하면 전력 수요가 높은 시간대에 집중된 생산 공정의 일부를 상대적으로 수요가 낮은 시간대로 조정할 수 있다. 이를 통해 특정 시간대의 피크 전력량을 완화하고 전력 사용량을 보다 균등하게 분산할 수 있다. 다만 실제 생산 일정에 적용할 때에는 공정 순서, 납기, 인력 배치, 설비 가동 조건 등의 운영 제약을 함께 고려해야 한다.

In [ ]:
# 골짜기 채우기: 임계값 위의 초과량과 아래의 여유량이 같아지는 상한선 탐색
load = workday[selected].dropna().to_numpy(dtype=float)
if len(load) == 0:
    raise ValueError("09:00~18:00 구간에 사용할 예측 전력량이 없습니다.")

lower_bound = float(load.min())
upper_bound = float(load.max())

def balance(threshold):
    """임계값 위 초과량 - 임계값 아래 여유량을 반환합니다."""
    above = np.maximum(load - threshold, 0).sum()
    below = np.maximum(threshold - load, 0).sum()
    return above - below

# balance(threshold)는 threshold가 커질수록 감소하므로 이진 탐색 적용
for _ in range(60):
    threshold = (lower_bound + upper_bound) / 2
    if balance(threshold) > 0:
        lower_bound = threshold
    else:
        upper_bound = threshold

threshold = (lower_bound + upper_bound) / 2
above = np.maximum(load - threshold, 0).sum()
below = np.maximum(threshold - load, 0).sum()

# 상한선 교차 시각과 방향을 계산합니다.
timestamps = workday["ts"].to_numpy()
values = workday[selected].to_numpy(dtype=float)
differences = values - threshold
tolerance = 1e-9
crossing_info = []

# 두 관측값 사이에서 상한선을 통과하는 경우
for index in range(len(values) - 1):
    left_diff = differences[index]
    right_diff = differences[index + 1]
    if left_diff * right_diff < 0:
        ratio = -left_diff / (right_diff - left_diff)
        start = pd.Timestamp(timestamps[index])
        end = pd.Timestamp(timestamps[index + 1])
        crossing_time = start + (end - start) * ratio
        direction = "상한선 위 -> 아래" if left_diff > 0 else "상한선 아래 -> 위"
        crossing_info.append((crossing_time, direction))

# 관측값 자체가 상한선과 같은 경우: 앞뒤 상태로 위->위, 아래->아래 등을 판정
for index, difference in enumerate(differences):
    if not np.isclose(difference, 0, atol=tolerance):
        continue

    before = None
    after = None
    previous = index - 1
    while previous >= 0 and np.isclose(differences[previous], 0, atol=tolerance):
        previous -= 1
    following = index + 1
    while following < len(differences) and np.isclose(differences[following], 0, atol=tolerance):
        following += 1
    if previous >= 0:
        before = "위" if differences[previous] > 0 else "아래"
    if following < len(differences):
        after = "위" if differences[following] > 0 else "아래"
    if before is not None and after is not None:
        crossing_info.append((pd.Timestamp(timestamps[index]), f"상한선 {before} -> {after}"))

# 같은 시각의 중복 교차점을 제거합니다.
crossing_info = sorted(set(crossing_info), key=lambda item: item[0])
crossing_times = [time for time, _ in crossing_info]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(workday["ts"], workday[selected], color="tab:blue", lw=1.5, label=f"예측 ({selected})")
ax.axhline(threshold, color="tab:red", ls="--", lw=1.5, label=f"골짜기 채우기 상한선 ({threshold:.1f} kW)")
ax.fill_between(workday["ts"], workday[selected], threshold,
                where=values > threshold, interpolate=True,
                color="tab:red", alpha=0.18, label="상한선 초과량")
ax.fill_between(workday["ts"], workday[selected], threshold,
                where=values < threshold, interpolate=True,
                color="tab:green", alpha=0.18, label="상한선 아래 여유량")
if crossing_times:
    ax.scatter(crossing_times, [threshold] * len(crossing_times),
               color="black", s=45, zorder=5, label="상한선 교차점")
ax.set_title(f"골짜기 채우기 상한선: {TEST_DATE:%Y-%m-%d} ({DOW_KR[TEST_DATE.dayofweek + 1]})")
ax.set_xlabel("시간")
ax.set_ylabel("15분 수요 (kW)")
ax.set_xlim(TEST_DATE + pd.Timedelta(hours=9), TEST_DATE + pd.Timedelta(hours=18))
ax.xaxis.set_major_locator(plt.matplotlib.dates.HourLocator(interval=1))
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%H:%M"))
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8, loc="upper right")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"찾은 상한선: {threshold:.2f} kW")
print(f"상한선 위 초과량: {above * 0.25:,.2f} kWh")
print(f"상한선 아래 여유량: {below * 0.25:,.2f} kWh")
print(f"차이: {(above - below) * 0.25:,.8f} kWh")
if crossing_info:
    print("상한선 교차 시각 및 방향:")
    for crossing_time, direction in crossing_info:
        print(f"- {crossing_time:%H:%M}: {direction}")
else:
    print("상한선과 만나는 지점이 없습니다.")

### 2. 야간 시간대 활용 방안

주간 시간대의 전력 사용량이 높고 피크 구간을 완화한 이후에도 전력 수요가 상대적으로 높게 유지될 것으로 예측된다면, 생산 공정의 일부를 밤이나 새벽 시간대로 분산하는 방안도 고려할 수 있다. 이를 통해 주간 시간대의 전력 수요를 추가로 낮추고 전체적인 부하를 분산할 수 있다.

다만 밤이나 새벽 시간대에는 인건비가 주간보다 1.5배 높게 적용되므로, 전력 피크 감소로 얻는 비용 절감 효과가 추가 인건비를 상쇄할 수 있는지 먼저 검토해야 한다. 또한 공장별 생산 공정, 설비 가동 조건, 교대 인력 확보, 안전 관리, 납기 등의 실제 운영 상황이 서로 다르므로 모든 공장에 동일한 방식으로 적용하기는 어렵다. 따라서 야간 생산 전환은 전력 절감 효과와 운영 비용 및 현장 제약을 함께 비교한 후 신중하게 결정해야 한다.

#### 간단한 비용 변화 예시

전력량과 공장 인원수가 선형 비례한다고 가정하고, 1년 내 기존 피크 전력량이 `180`, 전력량이 몰린 날에 주간 피크 전력량이 `200`, 주간 공장 인원이 `10명`이라고 하자. 전기 단가는 `C`, 주간 근로자 1명당 인건비는 `S`로 둔다.

- 기존 비용
  - 피크 기준 전기요금: $200C$
  - 인건비: $10S$

주간 생산량의 30%를 야간으로 이동해 주간 전력량을 30% 낮춘다고 가정하면, 주간 전력량은 `140`이 되고 야간으로 이동한 생산량은 `60`이 된다. 선형 비례 가정에 따라 주간 인원 3명에 해당하는 생산량을 야간으로 옮기면 다음과 같다.

- 생산량을 유지하면서 30%를 야간으로 이동한 경우
  - 피크 기준 전기요금 $180C$
  - 주간 인건비: $7S$
  - 야간 인건비: $3 \times 1.5S = 4.5S$
  - 총 인건비: $7S + 4.5S = 11.5S$

주간 내 피크 전력량은 `200`에서 `140`으로 30% 낮아져 최대수요전력에 따른 기본요금 부담을 줄일 가능성이 있다. 반면 총 인건비는 `10S`에서 `11.5S`로 증가하므로, 야간 전환은 피크 요금 절감액과 추가 인건비를 비교해 결정해야 한다.